[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_02/SFM_ch2_student_t/SFM_ch2_student_t.ipynb)

# SFM_ch2_student_t

Student-t Distribution for Financial Returns
Description:
- Student-t PDFs for various degrees of freedom
- MLE fit of Student-t to S&P 500 log-returns
- Log-scale density comparison (Student-t vs Normal)
- AIC/BIC model selection criteria
- Tail probability comparison
Statistics of Financial Markets course

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Chart style settings - Nature journal quality
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 8
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['lines.linewidth'] = 0.75

# Color palette
MAIN_BLUE = '#1A3A6E'
CRIMSON = '#DC3545'
FOREST = '#2E7D32'
AMBER = '#B5853F'
ORANGE = '#E67E22'

# Output directory
CHART_DIR = os.path.join('..', '..', '..', 'charts')
os.makedirs(CHART_DIR, exist_ok=True)

def save_fig(name):
    """Save figure with transparent background."""
    plt.savefig(os.path.join(CHART_DIR, f'{name}.pdf'),
                bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(CHART_DIR, f'{name}.png'),
                bbox_inches='tight', transparent=True, dpi=300)
    plt.close()
    print(f"   Saved: {name}.pdf/.png")

## Student-t PDFs

In [ ]:
x = np.linspace(-6, 6, 500)

# Define distributions
dfs = [3, 5, 10, 30]
colors_df = [CRIMSON, ORANGE, FOREST, AMBER]
labels_df = [f'$t_{{{d}}}$' for d in dfs]

fig, axes = plt.subplots(1, 2, figsize=(7, 3))

# --- Left panel: linear scale ---
ax = axes[0]
for d, c, lab in zip(dfs, colors_df, labels_df):
    ax.plot(x, stats.t.pdf(x, df=d), color=c, linewidth=1.0, label=lab)
ax.plot(x, stats.norm.pdf(x), color=MAIN_BLUE, linewidth=1.2,
        linestyle='--', label='Normal')
ax.set_xlabel('$x$')
ax.set_ylabel('Density')
ax.set_title('Linear scale')
ax.set_xlim(-6, 6)
ax.set_ylim(0, None)
ax.legend(loc='upper right', fontsize=7, frameon=False)

# --- Right panel: log scale ---
ax = axes[1]
for d, c, lab in zip(dfs, colors_df, labels_df):
    ax.plot(x, stats.t.pdf(x, df=d), color=c, linewidth=1.0, label=lab)
ax.plot(x, stats.norm.pdf(x), color=MAIN_BLUE, linewidth=1.2,
        linestyle='--', label='Normal')
ax.set_yscale('log')
ax.set_xlabel('$x$')
ax.set_ylabel('Density (log scale)')
ax.set_title('Log scale')
ax.set_xlim(-6, 6)
ax.set_ylim(1e-5, 1)
ax.legend(loc='upper right', fontsize=7, frameon=False)

plt.tight_layout()
save_fig('ch2_student_t_pdfs')

## Download Data and Fit

In [ ]:
# Download S&P 500 data
data = yf.download('^GSPC', start='2000-01-01', end='2025-01-01', progress=False)
close = data['Close'].squeeze()
log_ret = np.log(close / close.shift(1)).dropna()

print(f"   Period: {close.index[0].strftime('%Y-%m-%d')} to {close.index[-1].strftime('%Y-%m-%d')}")
print(f"   Observations: {len(log_ret)}")
print(f"   Mean: {log_ret.mean():.6f}")
print(f"   Std:  {log_ret.std():.6f}")
print()

# MLE fit: Student-t
t_params = stats.t.fit(log_ret)
t_df, t_loc, t_scale = t_params
print(f"   Student-t MLE fit:")
print(f"     df    = {t_df:.2f}")
print(f"     loc   = {t_loc:.6f}")
print(f"     scale = {t_scale:.6f}")
print()

# MLE fit: Normal
n_loc, n_scale = stats.norm.fit(log_ret)
print(f"   Normal MLE fit:")
print(f"     loc   = {n_loc:.6f}")
print(f"     scale = {n_scale:.6f}")
print()

# Log-likelihood, AIC, BIC
n = len(log_ret)
ll_t = np.sum(stats.t.logpdf(log_ret, *t_params))
ll_n = np.sum(stats.norm.logpdf(log_ret, n_loc, n_scale))

k_t = 3  # df, loc, scale
k_n = 2  # loc, scale

aic_t = -2 * ll_t + 2 * k_t
bic_t = -2 * ll_t + k_t * np.log(n)
aic_n = -2 * ll_n + 2 * k_n
bic_n = -2 * ll_n + k_n * np.log(n)

print(f"   {'Model':<12} {'LogLik':>12} {'AIC':>12} {'BIC':>12}")
print(f"   {'-'*48}")
print(f"   {'Student-t':<12} {ll_t:>12.2f} {aic_t:>12.2f} {bic_t:>12.2f}")
print(f"   {'Normal':<12} {ll_n:>12.2f} {aic_n:>12.2f} {bic_n:>12.2f}")
print()
print(f"   Student-t wins by AIC: {aic_n - aic_t:.1f}")
print(f"   Student-t wins by BIC: {bic_n - bic_t:.1f}")

## Student-t vs Normal Fit

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))

# Histogram (log y-axis)
counts, bins, _ = ax.hist(log_ret, bins=150, density=True, alpha=0.4,
                          color=MAIN_BLUE, edgecolor='none', label='S&P 500')

# Fitted densities
x_fit = np.linspace(log_ret.min() * 1.1, log_ret.max() * 1.1, 500)
ax.plot(x_fit, stats.t.pdf(x_fit, *t_params), color=CRIMSON, linewidth=1.2,
        label=f'Student-t ($\\nu$={t_df:.1f})')
ax.plot(x_fit, stats.norm.pdf(x_fit, n_loc, n_scale), color=FOREST,
        linewidth=1.2, linestyle='--', label='Normal')

ax.set_yscale('log')
ax.set_xlabel('Log-return')
ax.set_ylabel('Density (log scale)')
ax.set_ylim(1e-1, None)
ax.legend(loc='upper right', fontsize=7, frameon=False)

# Annotate
ax.text(0.02, 0.95, f'Fitted $\\nu$ = {t_df:.2f}\nAIC gain = {aic_n - aic_t:.0f}',
        transform=ax.transAxes, fontsize=7, va='top', color=CRIMSON)

plt.tight_layout()
save_fig('ch2_student_t_fit')

## Tail Probability Comparison

In [ ]:
# Tail probability comparison: P(|X| > k * sigma)
sigma_emp = log_ret.std()
mu_emp = log_ret.mean()

ks = [3, 4, 5, 6]
tail_student = []
tail_normal = []
tail_empirical = []

for k in ks:
    threshold = k * sigma_emp
    # Student-t: P(|X - loc| > k*sigma) using fitted params
    p_t = stats.t.sf(mu_emp + threshold, *t_params) + stats.t.cdf(mu_emp - threshold, *t_params)
    # Normal: P(|X - mu| > k*sigma)
    p_n = stats.norm.sf(mu_emp + threshold, n_loc, n_scale) + stats.norm.cdf(mu_emp - threshold, n_loc, n_scale)
    # Empirical
    p_e = np.mean(np.abs(log_ret - mu_emp) > threshold)
    tail_student.append(p_t)
    tail_normal.append(p_n)
    tail_empirical.append(p_e)

# Print table
print(f"   {'k':>4}  {'Student-t':>12}  {'Normal':>12}  {'Empirical':>12}  {'t/Normal':>10}")
print(f"   {'-'*56}")
for i, k in enumerate(ks):
    ratio = tail_student[i] / tail_normal[i] if tail_normal[i] > 0 else np.inf
    print(f"   {k:>4}  {tail_student[i]:>12.6f}  {tail_normal[i]:>12.6f}  {tail_empirical[i]:>12.6f}  {ratio:>10.1f}x")

# Bar chart
fig, ax = plt.subplots(figsize=(7, 3))

x_pos = np.arange(len(ks))
width = 0.25

bars_t = ax.bar(x_pos - width, tail_student, width, color=CRIMSON, label='Student-t', alpha=0.85)
bars_n = ax.bar(x_pos, tail_normal, width, color=MAIN_BLUE, label='Normal', alpha=0.85)
bars_e = ax.bar(x_pos + width, tail_empirical, width, color=FOREST, label='Empirical', alpha=0.85)

ax.set_yscale('log')
ax.set_xlabel('$k$ (multiples of $\\sigma$)')
ax.set_ylabel('$P(|X - \\mu| > k\\sigma)$')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{k}$\\sigma$' for k in ks])
ax.legend(loc='upper right', fontsize=7, frameon=False)

plt.tight_layout()
save_fig('ch2_student_t_tails')